In [1]:
import os
import gc
import random
import shutil

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from datasets import Dataset


from transformers import (
AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled!")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

In [4]:
DATASET1_PATH = "/kaggle/input/datasets/hamedmohammedadel/nlp-final-term-processedtext/dataset1_processed.csv"

DATASET2_PATH = "/kaggle/input/datasets/hamedmohammedadel/nlp-final-term-processedtext/dataset2_processed.csv"

In [5]:
df1 = pd.read_csv(DATASET1_PATH)
df2 = pd.read_csv(DATASET2_PATH)

print("Dataset 1 Shape:", df1.shape)
display(df1.head())

print("\nDataset 2 Shape:", df2.shape)
display(df2.head())

Dataset 1 Shape: (5000, 3)


,text,author,processed_text
0,The Internet continued to grow in leaps and bo...,AaronPressman,internet continue grow leap bound year online ...
1,Elementary school students with access to the ...,AaronPressman,elementary school student access internet lear...
2,Hewlett-Packard Co. unveiled a new plan Monday...,AaronPressman,hewlett packard co. unveil new plan monday boo...
3,A U.S. District Court judge Wednesday barred n...,AaronPressman,u.s district court judge wednesday bar new fed...
4,With the Clinton administration busily reworki...,AaronPressman,clinton administration busily rework 1995 prop...



Dataset 2 Shape: (19576, 3)


,text,author,processed_text
0,"This process, however, afforded me no means of...",EAP,process however afford I mean ascertain dimens...
1,It never once occurred to me that the fumbling...,HPL,never occur I fumbling might mere mistake
2,"In his left hand was a gold snuff box, from wh...",EAP,left hand gold snuff box caper hill cut manner...
3,How lovely is spring As we looked from Windsor...,MWS,lovely spring look windsor terrace sixteen fer...
4,"Finding nothing else, not even gold, the Super...",HPL,find nothing else even gold superintendent aba...


In [6]:
print("Dataset 1 Train columns:")
print(df1.columns.tolist())


print("\nDataset 2 columns:")
print(df2.columns.tolist())

Dataset 1 Train columns:
['text', 'author', 'processed_text']

Dataset 2 columns:
['text', 'author', 'processed_text']


In [7]:
TEXT_COLUMN = "processed_text"
LABEL_COLUMN = "author"

In [8]:
df1 = df1[[TEXT_COLUMN, LABEL_COLUMN]].copy()

df2 = df2[[TEXT_COLUMN, LABEL_COLUMN]].copy()

df1.columns = ["processed_text", "author"]
df2.columns = ["processed_text", "author"]

In [9]:
def clean_dataset(df):

    df = df.copy()

    df = df.dropna(subset=["processed_text", "author"])

    df["processed_text"] = (df["processed_text"].astype(str).str.strip())

    df = df[df["processed_text"] != ""]

    df = df.reset_index(drop=True)

    return df

In [10]:
df1 = clean_dataset(df1)
df2 = clean_dataset(df2)

print("Dataset 1:", df1.shape)
print("Dataset 2:", df2.shape)

Dataset 1: (5000, 2)
Dataset 2: (19576, 2)


In [11]:
print("DATASET 1 AUTHOR DISTRIBUTION")
print(df1["author"].value_counts())
print("\n" + "=" * 60)


print("\nDATASET 2 AUTHOR DISTRIBUTION")
print(df2["author"].value_counts())

DATASET 1 AUTHOR DISTRIBUTION
author
AaronPressman        100
AlanCrosby           100
AlexanderSmith       100
BenjaminKangLim      100
BernardHickey        100
BradDorfman          100
DarrenSchuettler     100
DavidLawder          100
EdnaFernandes        100
EricAuchard          100
FumikoFujisaki       100
GrahamEarnshaw       100
HeatherScoffield     100
JanLopatka           100
JaneMacartney        100
JimGilchrist         100
JoWinterbottom       100
JoeOrtiz             100
JohnMastrini         100
JonathanBirt         100
KarlPenhaul          100
KeithWeir            100
KevinDrawbaugh       100
KevinMorrison        100
KirstinRidley        100
KouroshKarimkhany    100
LydiaZajc            100
LynneO'Donnell       100
LynnleyBrowning      100
MarcelMichelson      100
MarkBendeich         100
MartinWolk           100
MatthewBunce         100
MichaelConnor        100
MureDickie           100
NickLouth            100
PatriciaCommins      100
PeterHumphrey        100
PierreTran   

In [12]:
def encode_labels(df):

    df = df.copy()

    label_encoder = LabelEncoder()

    df["label"] = label_encoder.fit_transform(df["author"])

    return df, label_encoder

In [13]:
df1, le1 = encode_labels(df1)
df2, le2 = encode_labels(df2)

In [14]:
print("DATASET 1 AUTHOR MAPPING")

for i, author in enumerate(le1.classes_):
    print(f"{i} → {author}")
    


print("\nDATASET 2 AUTHOR MAPPING")

for i, author in enumerate(le2.classes_):
    print(f"{i} → {author}")

DATASET 1 AUTHOR MAPPING
0 → AaronPressman
1 → AlanCrosby
2 → AlexanderSmith
3 → BenjaminKangLim
4 → BernardHickey
5 → BradDorfman
6 → DarrenSchuettler
7 → DavidLawder
8 → EdnaFernandes
9 → EricAuchard
10 → FumikoFujisaki
11 → GrahamEarnshaw
12 → HeatherScoffield
13 → JanLopatka
14 → JaneMacartney
15 → JimGilchrist
16 → JoWinterbottom
17 → JoeOrtiz
18 → JohnMastrini
19 → JonathanBirt
20 → KarlPenhaul
21 → KeithWeir
22 → KevinDrawbaugh
23 → KevinMorrison
24 → KirstinRidley
25 → KouroshKarimkhany
26 → LydiaZajc
27 → LynneO'Donnell
28 → LynnleyBrowning
29 → MarcelMichelson
30 → MarkBendeich
31 → MartinWolk
32 → MatthewBunce
33 → MichaelConnor
34 → MureDickie
35 → NickLouth
36 → PatriciaCommins
37 → PeterHumphrey
38 → PierreTran
39 → RobinSidel
40 → RogerFillion
41 → SamuelPerry
42 → SarahDavison
43 → ScottHillis
44 → SimonCowell
45 → TanEeLyn
46 → TheresePoletti
47 → TimFarrand
48 → ToddNissen
49 → WilliamKazer

DATASET 2 AUTHOR MAPPING
0 → EAP
1 → HPL
2 → MWS


In [15]:
def create_splits(df):

    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])

    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"])

    return (train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True))

In [16]:
train_df1, val_df1, test_df1 = create_splits(df1)

print("Dataset 1 Splits")

print("Train:", train_df1.shape)
print("Validation:", val_df1.shape)
print("Test:", test_df1.shape)

Dataset 1 Splits
Train: (3500, 3)
Validation: (750, 3)
Test: (750, 3)


In [17]:
train_df2, val_df2, test_df2 = create_splits(df2)

print("Dataset 2 Splits")

print("Train:", train_df2.shape)
print("Validation:", val_df2.shape)
print("Test:", test_df2.shape)

Dataset 2 Splits
Train: (13703, 3)
Validation: (2936, 3)
Test: (2937, 3)


In [18]:
train_df1.to_csv( "/kaggle/working/dataset1_train.csv",index=False)

val_df1.to_csv( "/kaggle/working/dataset1_validation.csv", index=False)

test_df1.to_csv( "/kaggle/working/dataset1_test.csv", index=False)


train_df2.to_csv("/kaggle/working/dataset2_train.csv",index=False)

val_df2.to_csv("/kaggle/working/dataset2_validation.csv",index=False)

test_df2.to_csv("/kaggle/working/dataset2_test.csv",index=False)

print("All dataset splits saved!")

All dataset splits saved!


In [19]:
MODEL_NAME = "bert-base-uncased"

MAX_LENGTH = 128

EPOCHS = 5

WARMUP_RATIO = 0.1

DROPOUT = 0.1

EARLY_STOPPING_PATIENCE = 2

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [21]:
sample_text = train_df1["processed_text"].iloc[0]

print("Processed Text:\n")
print(sample_text)

encoded = tokenizer(sample_text,truncation=True, max_length=MAX_LENGTH)

print("\nInput IDs:")
print(encoded["input_ids"][:20])

print("\nAttention Mask:")
print(encoded["attention_mask"][:20])

print("\nBERT Tokens:")
print(tokenizer.tokenize(sample_text)[:30])

Processed Text:

second time row united auto worker union choose ford motor co. lead union 's labor contract negotiation big three u.s automaker official say late tuesday uaw vice president ernest lofton say union leader notify ford executive decision tuesday add union view company traditional strike target '' talk lead company regard resolve 1996 negotiation '' lofton say tuesday night brief interview wdiv tv selection allow 2 u.s automaker negotiate agreement serve pattern rival general motors corp chrysler corp follow contract cover nearly 400,000 u.s hourly worker three company expire september 14 despite conciliatory talk ford target company face threat national strike fail reach agreement expiration ford spokesman jon harmon would read follow statement tuesday hard work negotiate uaw subcommittee level well main table encourage tone negotiation '' say team work assumption closely involve whatever transpire next two week preference along '' decision mark second time row union choo

In [22]:
def create_hf_dataset(df):

    dataset = Dataset.from_pandas(
        df[
            ["processed_text", "label"]
        ],
        preserve_index=False
    )

    return dataset

In [23]:
def tokenize_function(examples):

    return tokenizer(
        examples["processed_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [24]:
train_dataset1 = create_hf_dataset(train_df1)
val_dataset1 = create_hf_dataset(val_df1)
test_dataset1 = create_hf_dataset(test_df1)

In [25]:
tokenized_train1 = train_dataset1.map(
    tokenize_function,
    batched=True
)

tokenized_val1 = val_dataset1.map(
    tokenize_function,
    batched=True
)

tokenized_test1 = test_dataset1.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [26]:
train_dataset2 = create_hf_dataset(train_df2)
val_dataset2 = create_hf_dataset(val_df2)
test_dataset2 = create_hf_dataset(test_df2)

In [27]:
tokenized_train2 = train_dataset2.map(
    tokenize_function,
    batched=True
)

tokenized_val2 = val_dataset2.map(
    tokenize_function,
    batched=True
)

tokenized_test2 = test_dataset2.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/13703 [00:00<?, ? examples/s]

Map:   0%|          | 0/2936 [00:00<?, ? examples/s]

Map:   0%|          | 0/2937 [00:00<?, ? examples/s]

In [28]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [29]:
NUM_LABELS_1 = len(le1.classes_)
NUM_LABELS_2 = len(le2.classes_)

print("Dataset 1 Authors:", NUM_LABELS_1)
print("Dataset 2 Authors:", NUM_LABELS_2)

Dataset 1 Authors: 50
Dataset 2 Authors: 3


In [30]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision = precision_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
experiment_configs = [

    
    {
        "learning_rate": 2e-5,
        "batch_size": 16,
        "weight_decay": 0.01
    },

    
    {
        "learning_rate": 3e-5,
        "batch_size": 16,
        "weight_decay": 0.01
    },

    {
        "learning_rate": 2e-5,
        "batch_size": 32,
        "weight_decay": 0.01
    },

    {
        "learning_rate": 3e-5,
        "batch_size": 32,
        "weight_decay": 0.01
    },

    {
        "learning_rate": 2e-5,
        "batch_size": 16,
        "weight_decay": 0.1
    },


    {
        "learning_rate": 3e-5,
        "batch_size": 16,
        "weight_decay": 0.1
    },

    {
        "learning_rate": 2e-5,
        "batch_size": 32,
        "weight_decay": 0.1
    },

    {
        "learning_rate": 3e-5,
        "batch_size": 32,
        "weight_decay": 0.1
    }
]

In [32]:
pd.DataFrame(experiment_configs)

,learning_rate,batch_size,weight_decay
0,0.00002,16,0.01
1,0.00003,16,0.01
2,0.00002,32,0.01
3,0.00003,32,0.01
4,0.00002,16,0.10
5,0.00003,16,0.10
6,0.00002,32,0.10
7,0.00003,32,0.10


In [ ]:
WORKING_DIR = "/kaggle/working"

def train_one_experiment(

    train_dataset,
    val_dataset,
    test_dataset,
    num_labels,
    config,
    dataset_name,
    experiment_number
):

    print("\n" + "=" * 70)

    print(f"DATASET: {dataset_name}")
    print(f"EXPERIMENT: {experiment_number}")

    print(
        f"Learning Rate: "
        f"{config['learning_rate']}"
    )

    print(
        f"Batch Size: "
        f"{config['batch_size']}"
    )

    print(
        f"Weight Decay: "
        f"{config['weight_decay']}"
    )

    print("=" * 70)



    model = AutoModelForSequenceClassification.from_pretrained(

        MODEL_NAME,

        num_labels=num_labels,

        hidden_dropout_prob=DROPOUT,

        attention_probs_dropout_prob=DROPOUT
    )

    output_dir = os.path.join(

        WORKING_DIR,

        f"bert_{dataset_name}_exp_{experiment_number}"
    )



    training_args = TrainingArguments(

        output_dir=output_dir,

        learning_rate=config["learning_rate"],

        per_device_train_batch_size=config["batch_size"],

        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=EPOCHS,

        weight_decay=config["weight_decay"],

        warmup_ratio=WARMUP_RATIO,

        optim="adamw_torch",

        eval_strategy="epoch",

        save_strategy="epoch",

        logging_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model="f1",

        greater_is_better=True,

        save_total_limit=1,

        report_to="none",

        fp16=torch.cuda.is_available(),

        seed=SEED
    )


    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        eval_dataset=val_dataset,

        data_collator=data_collator,

        compute_metrics=compute_metrics,

        callbacks=[

            EarlyStoppingCallback(

                early_stopping_patience=
                EARLY_STOPPING_PATIENCE
            )
        ]
    )


    trainer.train()


    best_checkpoint = (
        trainer.state.best_model_checkpoint
    )

    best_metric = (
        trainer.state.best_metric
    )

    print("\nBEST CHECKPOINT:")
    print(best_checkpoint)

    print("\nBEST VALIDATION F1:")
    print(best_metric)

    validation_results = trainer.evaluate()



    val_prediction_output = trainer.predict(
        val_dataset
    )

    val_logits = (
        val_prediction_output.predictions
    )

    val_probabilities = torch.softmax(

        torch.tensor(val_logits),

        dim=1

    ).cpu().numpy()

    val_labels = (
        val_prediction_output.label_ids
    )



    test_prediction_output = trainer.predict(
        test_dataset
    )

    test_logits = (
        test_prediction_output.predictions
    )

    test_probabilities = torch.softmax(

        torch.tensor(test_logits),

        dim=1

    ).cpu().numpy()

    test_labels = (
        test_prediction_output.label_ids
    )

    prediction_dir = os.path.join(

        WORKING_DIR,

        "bert_predictions"
    )

    os.makedirs(
        prediction_dir,
        exist_ok=True
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_val_prob.npy"

        ),

        val_probabilities
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_val_labels.npy"

        ),

        val_labels
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_test_prob.npy"

        ),

        test_probabilities
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_test_labels.npy"

        ),

        test_labels
    )


    result = {

        "experiment": experiment_number,

        "learning_rate":
            config["learning_rate"],

        "batch_size":
            config["batch_size"],

        "weight_decay":
            config["weight_decay"],

        "best_validation_f1":
            best_metric,

        "accuracy":
            validation_results["eval_accuracy"],

        "precision":
            validation_results["eval_precision"],

        "recall":
            validation_results["eval_recall"],

        "f1":
            validation_results["eval_f1"]
    }


    print("\nVALIDATION RESULT:")

    print(result)



    del trainer

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    if os.path.exists(output_dir):

        shutil.rmtree(output_dir)

        print(
            f"\nDeleted checkpoint folder:"
            f"\n{output_dir}"
        )


    return result

In [34]:
def run_all_experiments(

    train_dataset,

    val_dataset,

    test_dataset,

    num_labels,

    dataset_name
):

    all_results = []


    for experiment_number, config in enumerate(

        experiment_configs,

        start=1
    ):

        result = train_one_experiment(

            train_dataset=train_dataset,

            val_dataset=val_dataset,

            test_dataset=test_dataset,

            num_labels=num_labels,

            config=config,

            dataset_name=dataset_name,

            experiment_number=experiment_number
        )


        all_results.append(result)


        results_df = pd.DataFrame(
            all_results
        )


        results_df.to_csv(

            os.path.join(

                WORKING_DIR,

                f"bert_"
                f"{dataset_name}_results.csv"

            ),

            index=False
        )


    return pd.DataFrame(
        all_results
    )

In [35]:
bert_dataset1_results = run_all_experiments(

    train_dataset=tokenized_train1,

    val_dataset=tokenized_val1,

    test_dataset=tokenized_test1,
    
    num_labels=NUM_LABELS_1,

    dataset_name="dataset1"
)


DATASET: dataset1
EXPERIMENT: 1
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.01


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,7.380529,6.072964,0.368000,0.339197,0.368000,0.309736
2,5.215166,4.392889,0.542667,0.561143,0.542667,0.495674
3,4.035135,3.632711,0.601333,0.628458,0.601333,0.568554
4,3.429062,3.276104,0.626667,0.638955,0.626667,0.598313
5,3.135966,3.154855,0.633333,0.638434,0.633333,0.607498


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (122 outputs left out)

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


BEST CHECKPOINT:
/kaggle/working/bert_dataset1_exp_8/checkpoint-275

BEST VALIDATION F1:
0.6221839227873254


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



VALIDATION RESULT:
{'experiment': 8, 'learning_rate': 3e-05, 'batch_size': 32, 'weight_decay': 0.1, 'best_validation_f1': 0.6221839227873254, 'accuracy': 0.652, 'precision': 0.6796737734588991, 'recall': 0.652, 'f1': 0.6221839227873254}

Deleted checkpoint folder:
/kaggle/working/bert_dataset1_exp_8


In [36]:
bert_dataset1_results.sort_values(
    by="f1",
    ascending=False
)

,experiment,learning_rate,batch_size,weight_decay,best_validation_f1,accuracy,precision,recall,f1
1,2,0.00003,16,0.01,0.659874,0.681333,0.699904,0.681333,0.659874
5,6,0.00003,16,0.10,0.651630,0.672000,0.685775,0.672000,0.651630
3,4,0.00003,32,0.01,0.631183,0.661333,0.668952,0.661333,0.631183
4,5,0.00002,16,0.10,0.622495,0.656000,0.655529,0.656000,0.622495
7,8,0.00003,32,0.10,0.622184,0.652000,0.679674,0.652000,0.622184
0,1,0.00002,16,0.01,0.607498,0.633333,0.638434,0.633333,0.607498
6,7,0.00002,32,0.10,0.555826,0.596000,0.580921,0.596000,0.555826
2,3,0.00002,32,0.01,0.550742,0.590667,0.599154,0.590667,0.552010


In [37]:
bert_dataset2_results = run_all_experiments(

    train_dataset=tokenized_train2,

    test_dataset=tokenized_test2,

    val_dataset=tokenized_val2,

    num_labels=NUM_LABELS_2,

    dataset_name="dataset2"
)


DATASET: dataset2
EXPERIMENT: 1
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.01


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.645806,1.247072,0.731948,0.742058,0.731948,0.731190
2,0.961265,1.066947,0.788147,0.792605,0.788147,0.788193
3,0.554821,1.095817,0.801431,0.803272,0.801431,0.801481
4,0.325016,1.338061,0.793597,0.797916,0.793597,0.793397
5,0.198735,1.489735,0.801431,0.804096,0.801431,0.801626


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (123 outputs left out)

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


BEST CHECKPOINT:
/kaggle/working/bert_dataset2_exp_8/checkpoint-1075

BEST VALIDATION F1:
0.8109652936469177


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



VALIDATION RESULT:
{'experiment': 8, 'learning_rate': 3e-05, 'batch_size': 32, 'weight_decay': 0.1, 'best_validation_f1': 0.8109652936469177, 'accuracy': 0.8109673024523161, 'precision': 0.8115244768368607, 'recall': 0.8109673024523161, 'f1': 0.8109652936469177}

Deleted checkpoint folder:
/kaggle/working/bert_dataset2_exp_8


In [38]:
bert_dataset1_results["dataset"] = "Dataset 1"

bert_dataset2_results["dataset"] = "Dataset 2"


bert_all_results = pd.concat(

    [
        bert_dataset1_results,

        bert_dataset2_results
    ],

    ignore_index=True
)


bert_all_results.to_csv(

    "/kaggle/working/"
    "bert_all_results.csv",

    index=False
)


display(
    bert_all_results
)

,experiment,learning_rate,batch_size,weight_decay,best_validation_f1,accuracy,precision,recall,f1,dataset
0,1,0.00002,16,0.01,0.607498,0.633333,0.638434,0.633333,0.607498,Dataset 1
1,2,0.00003,16,0.01,0.659874,0.681333,0.699904,0.681333,0.659874,Dataset 1
2,3,0.00002,32,0.01,0.550742,0.590667,0.599154,0.590667,0.552010,Dataset 1
3,4,0.00003,32,0.01,0.631183,0.661333,0.668952,0.661333,0.631183,Dataset 1
4,5,0.00002,16,0.10,0.622495,0.656000,0.655529,0.656000,0.622495,Dataset 1
5,6,0.00003,16,0.10,0.651630,0.672000,0.685775,0.672000,0.651630,Dataset 1
6,7,0.00002,32,0.10,0.555826,0.596000,0.580921,0.596000,0.555826,Dataset 1
7,8,0.00003,32,0.10,0.622184,0.652000,0.679674,0.652000,0.622184,Dataset 1
8,1,0.00002,16,0.01,0.801626,0.801431,0.804096,0.801431,0.801626,Dataset 2
9,2,0.00003,16,0.01,0.806877,0.806880,0.808443,0.806880,0.806877,Dataset 2


In [39]:
MODEL_NAME = "roberta-base"

MAX_LENGTH = 128

EPOCHS = 5

WARMUP_RATIO = 0.1

DROPOUT = 0.1

EARLY_STOPPING_PATIENCE = 2

In [40]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

RobertaTokenizer(name_or_path='roberta-base', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
}
)


In [41]:

sample_text = train_df1["processed_text"].iloc[0]

print("Processed Text:\n")
print(sample_text)

encoded = tokenizer(sample_text,truncation=True, max_length=MAX_LENGTH)

print("\nInput IDs:")
print(encoded["input_ids"][:20])

print("\nAttention Mask:")
print(encoded["attention_mask"][:20])

print("\nBERT Tokens:")
print(tokenizer.tokenize(sample_text)[:30])

Processed Text:

second time row united auto worker union choose ford motor co. lead union 's labor contract negotiation big three u.s automaker official say late tuesday uaw vice president ernest lofton say union leader notify ford executive decision tuesday add union view company traditional strike target '' talk lead company regard resolve 1996 negotiation '' lofton say tuesday night brief interview wdiv tv selection allow 2 u.s automaker negotiate agreement serve pattern rival general motors corp chrysler corp follow contract cover nearly 400,000 u.s hourly worker three company expire september 14 despite conciliatory talk ford target company face threat national strike fail reach agreement expiration ford spokesman jon harmon would read follow statement tuesday hard work negotiate uaw subcommittee level well main table encourage tone negotiation '' say team work assumption closely involve whatever transpire next two week preference along '' decision mark second time row union choo

In [42]:
def create_hf_dataset(df):

    dataset = Dataset.from_pandas(
        df[
            ["processed_text", "label"]
        ],
        preserve_index=False
    )

    return dataset

In [43]:
def tokenize_function(examples):

    return tokenizer(
        examples["processed_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [44]:
train_dataset1 = create_hf_dataset(train_df1)
val_dataset1 = create_hf_dataset(val_df1)
test_dataset1 = create_hf_dataset(test_df1)

In [45]:
tokenized_train1 = train_dataset1.map(
    tokenize_function,
    batched=True
)

tokenized_val1 = val_dataset1.map(
    tokenize_function,
    batched=True
)

tokenized_test1 = test_dataset1.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [46]:
train_dataset2 = create_hf_dataset(train_df2)
val_dataset2 = create_hf_dataset(val_df2)
test_dataset2 = create_hf_dataset(test_df2)

In [47]:
tokenized_train2 = train_dataset2.map(
    tokenize_function,
    batched=True
)

tokenized_val2 = val_dataset2.map(
    tokenize_function,
    batched=True
)

tokenized_test2 = test_dataset2.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/13703 [00:00<?, ? examples/s]

Map:   0%|          | 0/2936 [00:00<?, ? examples/s]

Map:   0%|          | 0/2937 [00:00<?, ? examples/s]

In [48]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [49]:
NUM_LABELS_1 = len(le1.classes_)
NUM_LABELS_2 = len(le2.classes_)

print("Dataset 1 Authors:", NUM_LABELS_1)
print("Dataset 2 Authors:", NUM_LABELS_2)

Dataset 1 Authors: 50
Dataset 2 Authors: 3


In [50]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision = precision_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
experiment_configs = [

    {
        "learning_rate": 2e-5,
        "batch_size": 16,
        "weight_decay": 0.01
    },

    {
        "learning_rate": 3e-5,
        "batch_size": 16,
        "weight_decay": 0.01
    },

    {
        "learning_rate": 2e-5,
        "batch_size": 32,
        "weight_decay": 0.01
    },

    {
        "learning_rate": 3e-5,
        "batch_size": 32,
        "weight_decay": 0.01
    },

    {
        "learning_rate": 2e-5,
        "batch_size": 16,
        "weight_decay": 0.1
    },

    {
        "learning_rate": 3e-5,
        "batch_size": 16,
        "weight_decay": 0.1
    },

    {
        "learning_rate": 2e-5,
        "batch_size": 32,
        "weight_decay": 0.1
    },

    {
        "learning_rate": 3e-5,
        "batch_size": 32,
        "weight_decay": 0.1
    }
]

In [52]:
pd.DataFrame(experiment_configs)

,learning_rate,batch_size,weight_decay
0,0.00002,16,0.01
1,0.00003,16,0.01
2,0.00002,32,0.01
3,0.00003,32,0.01
4,0.00002,16,0.10
5,0.00003,16,0.10
6,0.00002,32,0.10
7,0.00003,32,0.10


In [ ]:

def roberta_train_one_experiment(

    train_dataset,
    val_dataset,
    test_dataset,
    num_labels,
    config,
    dataset_name,
    experiment_number
):

    print("\n" + "=" * 70)

    print(f"DATASET: {dataset_name}")
    print(f"EXPERIMENT: {experiment_number}")

    print(
        f"Learning Rate: "
        f"{config['learning_rate']}"
    )

    print(
        f"Batch Size: "
        f"{config['batch_size']}"
    )

    print(
        f"Weight Decay: "
        f"{config['weight_decay']}"
    )

    print("=" * 70)



    model = AutoModelForSequenceClassification.from_pretrained(

        MODEL_NAME,

        num_labels=num_labels,

        hidden_dropout_prob=DROPOUT,

        attention_probs_dropout_prob=DROPOUT
    )



    output_dir = os.path.join(

        WORKING_DIR,

        f"roberta_{dataset_name}_exp_{experiment_number}"
    )



    training_args = TrainingArguments(

        output_dir=output_dir,

        learning_rate=config["learning_rate"],

        per_device_train_batch_size=config["batch_size"],

        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=EPOCHS,

        weight_decay=config["weight_decay"],

        warmup_ratio=WARMUP_RATIO,

        optim="adamw_torch",

        eval_strategy="epoch",

        save_strategy="epoch",

        logging_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model="f1",

        greater_is_better=True,

        save_total_limit=1,

        report_to="none",

        fp16=torch.cuda.is_available(),

        seed=SEED
    )



    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        eval_dataset=val_dataset,

        data_collator=data_collator,

        compute_metrics=compute_metrics,

        callbacks=[

            EarlyStoppingCallback(

                early_stopping_patience=
                EARLY_STOPPING_PATIENCE
            )
        ]
    )



    trainer.train()



    best_checkpoint = (
        trainer.state.best_model_checkpoint
    )

    best_metric = (
        trainer.state.best_metric
    )

    print("\nBEST CHECKPOINT:")
    print(best_checkpoint)

    print("\nBEST VALIDATION F1:")
    print(best_metric)



    validation_results = trainer.evaluate()



    val_prediction_output = trainer.predict(
        val_dataset
    )

    val_logits = (
        val_prediction_output.predictions
    )

    val_probabilities = torch.softmax(

        torch.tensor(val_logits),

        dim=1

    ).cpu().numpy()

    val_labels = (
        val_prediction_output.label_ids
    )



    test_prediction_output = trainer.predict(
        test_dataset
    )

    test_logits = (
        test_prediction_output.predictions
    )

    test_probabilities = torch.softmax(

        torch.tensor(test_logits),

        dim=1

    ).cpu().numpy()

    test_labels = (
        test_prediction_output.label_ids
    )


    prediction_dir = os.path.join(

        WORKING_DIR,

        "roberta_predictions"
    )

    os.makedirs(
        prediction_dir,
        exist_ok=True
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_val_prob.npy"

        ),

        val_probabilities
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_val_labels.npy"

        ),

        val_labels
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_test_prob.npy"

        ),

        test_probabilities
    )


    np.save(

        os.path.join(

            prediction_dir,

            f"{dataset_name}_exp_"
            f"{experiment_number}_test_labels.npy"

        ),

        test_labels
    )



    result = {

        "experiment": experiment_number,

        "learning_rate":
            config["learning_rate"],

        "batch_size":
            config["batch_size"],

        "weight_decay":
            config["weight_decay"],

        "best_validation_f1":
            best_metric,

        "accuracy":
            validation_results["eval_accuracy"],

        "precision":
            validation_results["eval_precision"],

        "recall":
            validation_results["eval_recall"],

        "f1":
            validation_results["eval_f1"]
    }


    print("\nVALIDATION RESULT:")

    print(result)


    del trainer

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    if os.path.exists(output_dir):

        shutil.rmtree(output_dir)

        print(
            f"\nDeleted checkpoint folder:"
            f"\n{output_dir}"
        )


    return result

In [54]:
def roberta_run_all_experiments(

    train_dataset,

    val_dataset,

    test_dataset,

    num_labels,

    dataset_name
):

    all_results = []


    for experiment_number, config in enumerate(

        experiment_configs,

        start=1
    ):

        result = roberta_train_one_experiment(

            train_dataset=train_dataset,

            val_dataset=val_dataset,

            test_dataset=test_dataset,

            num_labels=num_labels,

            config=config,

            dataset_name=dataset_name,

            experiment_number=experiment_number
        )


        all_results.append(result)


        results_df = pd.DataFrame(
            all_results
        )


        results_df.to_csv(

            os.path.join(

                WORKING_DIR,

                f"roberta_"
                f"{dataset_name}_results.csv"

            ),

            index=False
        )


    return pd.DataFrame(
        all_results
    )

In [55]:
roberta_dataset1_results = roberta_run_all_experiments(

    train_dataset=tokenized_train1,

    test_dataset=tokenized_test1,

    val_dataset=tokenized_val1,

    num_labels=NUM_LABELS_1,

    dataset_name="dataset1"
)


DATASET: dataset1
EXPERIMENT: 1
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.01


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/di

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,7.216994,5.318110,0.440000,0.464221,0.440000,0.390289
2,4.384062,3.438132,0.582667,0.606844,0.582667,0.555980
3,3.171383,2.792880,0.642667,0.653206,0.642667,0.616137
4,2.606000,2.514990,0.668000,0.677710,0.668000,0.656146
5,2.338304,2.429016,0.686667,0.687082,0.686667,0.673262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (120 outputs left out)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


BEST CHECKPOINT:
/kaggle/working/roberta_dataset1_exp_8/checkpoint-275

BEST VALIDATION F1:
0.6578866059321966


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



VALIDATION RESULT:
{'experiment': 8, 'learning_rate': 3e-05, 'batch_size': 32, 'weight_decay': 0.1, 'best_validation_f1': 0.6578866059321966, 'accuracy': 0.672, 'precision': 0.6923420371338328, 'recall': 0.672, 'f1': 0.6578866059321966}

Deleted checkpoint folder:
/kaggle/working/roberta_dataset1_exp_8


In [56]:
roberta_dataset2_results = roberta_run_all_experiments(

    train_dataset=tokenized_train2,

    test_dataset=tokenized_test2,

    val_dataset=tokenized_val2,

    num_labels=NUM_LABELS_2,

    dataset_name="dataset2"
)


DATASET: dataset2
EXPERIMENT: 1
Learning Rate: 2e-05
Batch Size: 16
Weight Decay: 0.01


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/di

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.692661,1.268672,0.735354,0.735978,0.735354,0.735273
2,1.117862,1.227711,0.755450,0.761288,0.755450,0.755467
3,0.803268,1.214988,0.769074,0.774390,0.769074,0.768937
4,0.591679,1.319548,0.776907,0.779825,0.776907,0.776756
5,0.455727,1.355136,0.785082,0.786544,0.785082,0.785125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (123 outputs left out)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


BEST CHECKPOINT:
/kaggle/working/roberta_dataset2_exp_8/checkpoint-1075

BEST VALIDATION F1:
0.7888912591197479


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



VALIDATION RESULT:
{'experiment': 8, 'learning_rate': 3e-05, 'batch_size': 32, 'weight_decay': 0.1, 'best_validation_f1': 0.7888912591197479, 'accuracy': 0.7888283378746594, 'precision': 0.7911138290779585, 'recall': 0.7888283378746594, 'f1': 0.7888912591197479}

Deleted checkpoint folder:
/kaggle/working/roberta_dataset2_exp_8


In [57]:
!zip -r my_output.zip /kaggle/working/

from IPython.display import FileLink
FileLink('my_output.zip')


  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/dataset2_test.csv (deflated 60%)
  adding: kaggle/working/roberta_dataset2_results.csv (deflated 60%)
  adding: kaggle/working/bert_all_results.csv (deflated 66%)
  adding: kaggle/working/dataset1_validation.csv (deflated 64%)
  adding: kaggle/working/dataset1_train.csv (deflated 64%)
  adding: kaggle/working/roberta_predictions/ (stored 0%)
  adding: kaggle/working/roberta_predictions/dataset2_exp_8_test_prob.npy (deflated 8%)
  adding: kaggle/working/roberta_predictions/dataset1_exp_1_test_prob.npy (deflated 11%)
  adding: kaggle/working/roberta_predictions/dataset2_exp_3_test_labels.npy (deflated 94%)
  adding: kaggle/working/roberta_predictions/dataset2_exp_6_val_labels.npy (deflated 94%)
  adding: kaggle/working/roberta_predictions/dataset1_exp_3_test_labels.npy (deflated 81%)
  adding: kaggle/working/roberta_predictions/dataset2_exp_2_val_labels.npy (deflated 94%)
  adding: kaggle/working/roberta_predictions/dataset1_

/kaggle/working/my_output.zip

In [ ]:
def run_ensemble(

    dataset_name,

    bert_experiment,

    roberta_experiment
):


    bert_val_prob = np.load(

        f"{BERT_DIR}/"
        f"{dataset_name}_exp_"
        f"{bert_experiment}_val_prob.npy"
    )


    bert_val_labels = np.load(

        f"{BERT_DIR}/"
        f"{dataset_name}_exp_"
        f"{bert_experiment}_val_labels.npy"
    )



    roberta_val_prob = np.load(

        f"{ROBERTA_DIR}/"
        f"{dataset_name}_exp_"
        f"{roberta_experiment}_val_prob.npy"
    )


    bert_test_prob = np.load(

        f"{BERT_DIR}/"
        f"{dataset_name}_exp_"
        f"{bert_experiment}_test_prob.npy"
    )


    bert_test_labels = np.load(

        f"{BERT_DIR}/"
        f"{dataset_name}_exp_"
        f"{bert_experiment}_test_labels.npy"
    )


    roberta_test_prob = np.load(

        f"{ROBERTA_DIR}/"
        f"{dataset_name}_exp_"
        f"{roberta_experiment}_test_prob.npy"
    )



    X_meta_train = np.concatenate(

        [
            bert_val_prob,

            roberta_val_prob
        ],

        axis=1
    )


    X_meta_test = np.concatenate(

        [
            bert_test_prob,

            roberta_test_prob
        ],

        axis=1
    )


    y_meta_train = bert_val_labels

    y_test = bert_test_labels



    meta_learner = LogisticRegression(

        max_iter=1000,

        random_state=42
    )


    meta_learner.fit(

        X_meta_train,

        y_meta_train
    )



    predictions = meta_learner.predict(

        X_meta_test
    )
    
    accuracy = accuracy_score(

        y_test,

        predictions
    )


    precision = precision_score(

        y_test,

        predictions,

        average="weighted",

        zero_division=0
    )


    recall = recall_score(

        y_test,

        predictions,

        average="weighted",

        zero_division=0
    )


    f1 = f1_score(

        y_test,

        predictions,

        average="weighted",

        zero_division=0
    )


    return {

        "dataset": dataset_name,

        "bert_experiment":
            bert_experiment,

        "roberta_experiment":
            roberta_experiment,

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1
    }

In [ ]:
BERT_EXPERIMENT = 2

ROBERTA_EXPERIMENT = 8

ensemble_dataset1 = run_ensemble(

    dataset_name="dataset1",

    bert_experiment=BERT_EXPERIMENT,

    roberta_experiment=ROBERTA_EXPERIMENT
)


print(
    ensemble_dataset1
)


{'dataset': 'dataset1', 'bert_experiment': 8, 'roberta_experiment': 2, 'accuracy': 0.6733333333333333, 'precision': 0.6618608801976883, 'recall': 0.6733333333333333, 'f1': 0.6523246359404365}


In [ ]:
BERT_EXPERIMENT = 6

ROBERTA_EXPERIMENT = 2

ensemble_dataset2 = run_ensemble(

    dataset_name="dataset2",

    bert_experiment=BERT_EXPERIMENT,

    roberta_experiment=ROBERTA_EXPERIMENT
)


print(
    ensemble_dataset2
)


{'dataset': 'dataset2', 'bert_experiment': 8, 'roberta_experiment': 2, 'accuracy': 0.8352059925093633, 'precision': 0.8357543572420056, 'recall': 0.8352059925093633, 'f1': 0.8352018878133356}


In [ ]:
ensemble_results = pd.DataFrame(

    [
        ensemble_dataset1,

        ensemble_dataset2
    ]
)


display(
    ensemble_results
)


ensemble_results.to_csv(

    "/kaggle/working/"
    "final_ensemble_results.csv",

    index=False
)


,dataset,bert_experiment,roberta_experiment,accuracy,precision,recall,f1
0,dataset1,2,6,0.693333,0.697988,0.693333,0.675473
1,dataset2,8,2,0.835206,0.835754,0.835206,0.835202
